# NB03 — CLIMADA Hazard and Exposure Objects

This notebook converts the harmonised grids produced in NB02 into CLIMADA-native objects. In practice, it is the bridge between the gridded preprocessing workflow and the later mortality-impact notebooks: it turns daily `T2M` rasters into a `Hazard`, turns age-structured population rasters into `Exposures`, optionally attaches the Social Vulnerability Index (SVI), and writes the files that NB04 and the policy notebooks consume.

The key tasks in NB03 are:
- loading the shared grid metadata, city mask, hazard manifest, baseline population rasters, and optional vulnerability layer prepared in NB02;
- building one daily `T2M` hazard event per day across all target years saved in the manifest;
- correcting CLIMADA event frequencies so that each year's daily events sum to one expected year;
- constructing baseline age-differentiated exposure points on the same centroids, with age-group-specific impact-function IDs;
- saving the daily hazard, baseline exposure, and supporting event-summary tables;
- generating future scenario/year exposure objects by scaling the baseline rows with the scenario-specific grids saved in NB02.

NB03 does not define impact functions. Those are introduced in NB04. Its role is to make sure that hazard and exposure are represented in the exact format, geometry, and frequency convention that the downstream impact calculations require.


In [ ]:
import os
os.environ["URBAN_HEAT_OUTPUT_VARIANT"] = "masselot_main_agnostic"
os.environ["IF_MAIN_FAMILY"] = "masselot_tail"


In [ ]:
# City selector - the ONLY per-city line in this agnostic notebook.
# Set CITY to rome / athens / lisbon / copenhagen (any configured city).
import os
os.environ.setdefault("CITY", "Rome")


In [ ]:
# Bootstrap the city-specific configuration and resolve the working paths.
from pathlib import Path
import os, sys

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand/"cityheat").is_dir() and (cand/"configs").is_dir():
            return cand
    raise RuntimeError("Repo root not found.")
ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup_masselot_main import bootstrap
from cityheat.paths import make_P, ensure_out

# `CITY` is the intended selector; the fallback below is only a notebook-local default.
SLUG = globals().get("SLUG", os.environ["CITY"]).lower()

C    = bootstrap(SLUG)      # reads configs/<slug>.yml and syncs that city only if wanted
CFG  = C["CFG"]; CITY = C["CITY"]
BASE = C["BASE"]; OUT = C["OUT"]; INT = C["INT"]

P    = make_P(BASE)         # read-only path helper
OUTP = ensure_out(OUT)      # write-safe path helper
print(f"→ City: {CITY}  |  BASE={BASE}  OUT={OUT}  INT={INT}")


In [ ]:
cfg  = C.get("cfg")     # parsed YAML config


In [ ]:
# Read the NB02 manifests and confirm that the baseline grid products exist before proceeding.
import json
from pathlib import Path

with open(INT / "exposure_manifest.json", "r") as f:
    expo_manifest = json.load(f)

BASE_POP_YEAR = 2020

age_npz = Path(expo_manifest["direct_worldpop"][str(BASE_POP_YEAR)]["age_npz"])
pop_npz = Path(expo_manifest["direct_worldpop"][str(BASE_POP_YEAR)]["pop_npz"])

needed_nb2 = [
    INT / "template_ref.tif",
    INT / "city_mask.tif",
    age_npz,
    pop_npz,
    OUT / f"{SLUG}_fua.gpkg",
    INT / "hazard_manifest.json",
    INT / "exposure_manifest.json",
]

missing = [p for p in needed_nb2 if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Re-run Notebook 2. Missing:\n" + "\n".join(map(str, missing))
    )


## Load NB02 Outputs and Core Packages

The notebook starts by loading the shared assets created in NB02: the reference grid, the city mask, the baseline population rasters, the FUA geometry, the hazard manifest, and the exposure manifest. If the vulnerability layer was produced in NB02, it is also loaded here so it can be attached to the exposure points later.


In [ ]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio as rio
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import array_bounds

from climada.hazard import Hazard, Centroids
from scipy import sparse

plt.rcParams.update({"figure.dpi": 130})

In [ ]:
from pathlib import Path
import json, numpy as np, rasterio as rio
import geopandas as gpd

# Load the reference grid, city mask, baseline population rasters, FUA geometry, and hazard manifest.
tmpl_path = INT / "template_ref.tif"
mask_tif  = INT / "city_mask.tif"
mask_npz  = INT / "city_mask.npz"
fua_gp    = OUT / f"{SLUG}_fua.gpkg"
manifest  = INT / "hazard_manifest.json"
exp_meta  = INT / "exposure_meta.json"

# Reference-grid metadata
with rio.open(tmpl_path) as src:
    ref_meta = src.meta.copy()
    HGT, WDT = ref_meta["height"], ref_meta["width"]

# City mask: pixels inside the FUA with non-zero population in NB02.
if mask_tif.exists():
    with rio.open(mask_tif) as src:
        CITY_MASK = src.read(1).astype(bool)
elif mask_npz.exists():
    CITY_MASK = np.load(mask_npz)["city_mask"].astype(bool)
else:
    raise FileNotFoundError("city_mask.tif/npz not found. Re-run Notebook 2.")

assert CITY_MASK.shape == (HGT, WDT)
ref_data = np.where(CITY_MASK, 0.0, np.nan)

# Baseline age/pop rasters on the T2M reference grid.
agez = np.load(age_npz)
age_on_ref = {"<15": agez["lt15"], "15-64": agez["a15_64"], "65+": agez["g65"]}
pop_on_ref = np.load(pop_npz)["pop"]

# Exposure metadata
try:
    year = int(json.load(open(exp_meta))["worldpop_base_year"])
except Exception:
    year = 2020

# FUA geometry
if not fua_gp.exists():
    raise FileNotFoundError(f"{fua_gp.name} not found. Re-run Notebook 2 to save FUA.")
fua = gpd.read_file(fua_gp, layer="fua")
if fua.empty:
    raise ValueError("FUA is empty.")

# Daily T2M hazard manifest from NB02.
man_path = INT / "hazard_manifest.json"
man = json.load(open(man_path))
assert man.get("hazard_type") == "t2m_daily_mean", f"Unexpected hazard_type: {man.get('hazard_type')}"

years_all = sorted(map(int, man["files"].keys()))
years = [y for y in [2020, 2030, 2040, 2050] if y in years_all]
print("Using T2M years:", years)

t2m_paths = {int(y): Path(p) for y, p in man["files"].items()}


In [ ]:
# Optional vulnerability products from NB02; attach them later to exposures if available.
from cityheat.vulnerability_layer import (
    load_vulnerability,
    add_vulnerability_to_exposure_gdf,
)


In [ ]:
# Prefer the projected 2020 dynamic baseline; only fall back to the legacy
# unsuffixed bundle if projected files have not been built yet.
try:
    try:
        vuln = load_vulnerability(INT, SLUG, year=2020)
        vuln_label = "projected 2020 vulnerability baseline"
    except FileNotFoundError:
        vuln = load_vulnerability(INT, SLUG)
        vuln_label = "legacy baseline vulnerability bundle"
    svi_grid = vuln["svi"]          # 2D array on ref grid
    HAS_VULN = True
    print(f"Loaded {vuln_label}. "
          f"SVI mean={np.nanmean(svi_grid):.3f}, std={np.nanstd(svi_grid):.3f}")
    assert svi_grid.shape == (HGT, WDT), "SVI grid shape != ref grid shape"
except FileNotFoundError:
    svi_grid = None
    HAS_VULN = False
    print("No vulnerability layer found in INT – continuing without SVI.")


## CLIMADA Hazard: Daily Mean `T2M`


The hazard object is built from the daily `T2M` NetCDF files written in NB02. Each daily field becomes one CLIMADA event on the common UrbClim reference grid. The event metadata preserve the calendar date of each field so later notebooks can work at daily resolution.


In [ ]:
# Build the daily T2M hazard object, with one event per day.
import xarray as xr
from scipy import sparse
from pyproj import Transformer
from climada.hazard import Hazard, Centroids

# Convert the full reference-grid centroids from EPSG:3035 to WGS84 for CLIMADA.
# Compute one centroid per reference-grid cell.
# Sanity-check that the centroid count matches the grid size.
rows, cols = np.indices((HGT, WDT))
xs_m, ys_m = rio.transform.xy(ref_meta["transform"], rows, cols, offset="center")
x_flat = np.asarray(xs_m, float).ravel()   # meters (EPSG:3035)
y_flat = np.asarray(ys_m, float).ravel()
to_wgs84 = Transformer.from_crs(ref_meta["crs"], "EPSG:4326", always_xy=True)
lon, lat = to_wgs84.transform(x_flat, y_flat)
Ctd = Centroids(lat=np.asarray(lat, float), lon=np.asarray(lon, float))

assert (getattr(ref_meta["crs"], "to_epsg", lambda: None)() == 3035), "ref grid not EPSG:3035?"
assert len(Ctd.lat) == HGT*WDT and len(Ctd.lon) == HGT*WDT, "centroids length mismatch"

# Assemble one flattened intensity row per daily T2M field.
# The daily intensity matrix has shape (n_days x n_cells).
# Loop over every saved target year and every day in each file.
# Outside the city mask, intensities are stored as zero so the sparse matrix stays well defined.
# Each 2D field is flattened into one row before all days are stacked into a sparse matrix.
I_rows, dates, years_of = [], [], []
for y in years:
    da = xr.open_dataset(t2m_paths[y])["T2M"]  # (time,y,x) in °C, already masked to FUA in NB2
    for t in da.time.values:
        arr = da.sel(time=t).values
        arr = np.where(CITY_MASK, arr, np.nan)
        I_rows.append(np.nan_to_num(arr, nan=0.0).ravel().astype("float32"))
        d = np.datetime64(t, "D")
        dates.append(d)
        years_of.append(int(str(d)[:4]))

# Each row represents one daily temperature map.
I_daily = sparse.csr_matrix(np.vstack(I_rows))  # shape = (n_days_total, HGT*WDT)

# Build the daily affected-area fraction matrix.
# `fraction` repeats the 0/1 city mask for every day.
# In CLIMADA terms, cells inside the city have fraction 1 and cells outside have fraction 0.
mask_vec = CITY_MASK.ravel().astype(float)
F_daily = sparse.csr_matrix(np.tile(mask_vec, (I_daily.shape[0], 1)))

assert I_daily.shape[1] == HGT*WDT, "events x cells shape off"
assert F_daily.shape == I_daily.shape, "fraction footprint mismatch"

# Populate the CLIMADA Hazard object.
H = Hazard("T2M")
H.haz_type   = "T2M"
H.units      = "degC (daily mean)"
H.orig       = np.array([f"{CITY}_UrbClim_T2Mmean_daily"] * I_daily.shape[0], dtype=object)
H.event_id   = np.array([int(pd.Timestamp(d).strftime("%Y%m%d")) for d in dates], dtype=int)
H.event_name = np.array([f"T2M_{str(d)}" for d in dates], dtype=object)
H.date       = np.array(dates, dtype="datetime64[ns]")
H.frequency  = np.ones(I_daily.shape[0], dtype=float)
H.centroids  = Ctd
H.intensity  = I_daily
H.fraction   = F_daily

H.check()
print(H)


### Frequency Correction

For a daily heat hazard, the relevant unit is expected annual impact, not impact per unweighted event. We therefore replace CLIMADA's default event frequency with `1 / days_in_year`, so that the event frequencies sum to one within each year and annual impacts aggregate correctly across the full daily series.


In [ ]:
# Check year coverage, event counts, and calendar continuity before correcting frequencies.
years_of_arr = np.array(years_of, dtype=int)

print("Years included:", sorted(set(years_of_arr)))
print("Days per year:", {y: int((years_of_arr == y).sum()) for y in sorted(set(years_of_arr))})
print("First date:", dates[0], " | Last date:", dates[-1])

dts = np.array(dates, dtype="datetime64[D]")
diffs = np.diff(dts).astype("timedelta64[D]").astype(int)
print("Any gaps > 1 day?", np.any(diffs > 1), "| max gap:", diffs.max())

print("Hazard events:", H.size)
print("Hazard year range:", pd.to_datetime(H.date.min()).year, "-", pd.to_datetime(H.date.max()).year)


In [ ]:
# Replace the default CLIMADA event frequency by 1 / days_in_year for each daily event.
import numpy as np
import pandas as pd

# years for each event (one row per day)
years = pd.to_datetime(H.date).year.to_numpy()

def days_in_year(y): 
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365

H.frequency = np.array([1.0 / days_in_year(y) for y in years], dtype=float)
H.frequency_unit = "1/year"


In [ ]:
# Save a compact event table and verify that corrected frequencies sum to one within each year.
haz_df = pd.DataFrame({
    "date": pd.to_datetime(H.date),
    "event_id": H.event_id,
    "frequency": H.frequency,
})
haz_df.to_csv(INT / f"hazard_events_T2M_daily_{SLUG}.csv", index=False)

print("Saved:", INT / f"hazard_events_T2M_daily_{SLUG}.csv")
print(haz_df.groupby(haz_df["date"].dt.year)["frequency"].sum())


# Daily summary: mean of means per year should match yearly_summary
daily = pd.read_csv(INT / f"hazard_events_T2M_daily_{SLUG}.csv", parse_dates=["date"])
print(daily.groupby(daily["date"].dt.year)["frequency"].sum())  # again 1 each year


### Hazard Diagnostics

These plots are diagnostic checks on the assembled hazard object. They help verify spatial orientation, plausible seasonal values, and expected inter-year differences before impacts are computed from the daily fields.


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rasterio.transform import array_bounds

# Plot a single daily hazard field on the reference-grid extent (EPSG:3035).
T = ref_meta["transform"]
left, bottom, right, top = array_bounds(HGT, WDT, T)
extent = (left, right, bottom, top)

def plot_t2m_day(H, date_str):
    d = np.datetime64(date_str, 'D')
    idx = np.where(H.date.astype('datetime64[D]') == d)[0]
    if idx.size == 0:
        raise KeyError(f"No event for {date_str}")
    r = int(idx[0])

    arr = H.intensity.getrow(r).toarray().reshape(HGT, WDT)
    m = np.ma.masked_where(~CITY_MASK, arr)
    vmax = np.nanpercentile(m.compressed(), 99)

    plt.figure(figsize=(6.5, 6.2))
    im = plt.imshow(
        m,
        origin="lower",         
        extent=extent,          
        vmin=m.min(),
        vmax=vmax,
    )
    cb = plt.colorbar(im); cb.set_label("degC (daily mean)")
    plt.title(f"{CITY} — Daily mean T2M — {np.datetime_as_string(d, unit='D')}")
    plt.xlabel("ETRS89 / LAEA X (m)")
    plt.ylabel("ETRS89 / LAEA Y (m)")
    plt.gca().set_aspect('equal', 'box')
    plt.tight_layout(); plt.show()

plot_t2m_day(H, "2020-07-15")


In [ ]:
# Compute simple annual-mean maps from the daily hazard for a quick inter-year comparison.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def annual_mean_map(H, year):
    yrs = pd.to_datetime(H.date).year.values
    idx = np.where(yrs == year)[0]
    if idx.size == 0:
        raise ValueError(f"No events found for year={year}. Years available: {sorted(set(yrs))}")

    Iy = H.intensity[idx, :]  # (days in year) x (cells)
    mean_vec = np.array(Iy.sum(axis=0)).ravel() / len(idx)
    grid = mean_vec.reshape(HGT, WDT)
    return np.where(CITY_MASK, grid, np.nan)

years_to_plot = [2020, 2030, 2040, 2050]

maps = [annual_mean_map(H, y) for y in years_to_plot]
vmin = np.nanmin([np.nanmin(m) for m in maps])
vmax = np.nanmax([np.nanmax(m) for m in maps])

fig, axes = plt.subplots(1, len(years_to_plot), figsize=(18, 5), constrained_layout=True)

for ax, y, m in zip(axes, years_to_plot, maps):
    im = ax.imshow(m, extent=extent, origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(str(y))
    ax.set_axis_off()

cbar = fig.colorbar(im, ax=axes, shrink=0.85)
cbar.set_label("Annual mean daily T2M (°C)")
fig.suptitle(f"{CITY} — Hazard under SP scenario (annual mean)")
plt.show()


## CLIMADA Exposures: Age-Differentiated Population


Exposure points are created on the same centroids as the hazard grid. Each populated grid cell contributes one row per age group, with population in the `value` column and an age-group-specific `impf_T2M` identifier pointing to the impact functions defined in NB04. If the vulnerability layer is available, its continuous SVI value is attached to each exposure point after the baseline exposure is built.


In [ ]:
# Build the baseline exposure on the same centroids as the hazard grid.

# Only create exposure rows for valid city pixels.
haz_valid = CITY_MASK.ravel() # only create exposure points for cells inside FUA, and also only where population is > 0

# Age-group-specific impact-function IDs used later in NB04.
impf_id_T2M = {"<15": 1, "15-64": 2, "65+": 3}

# Create one exposure row per populated cell and age group.
frames = []
for grp, raster in age_on_ref.items():
    pop_flat = np.nan_to_num(raster, nan=0.0).ravel().astype(float)
    valid = haz_valid & (pop_flat > 0)
    if valid.any():
        frames.append(
            gpd.GeoDataFrame(
                {
                    "value":     pop_flat[valid],
                    "longitude": lon[valid],    # longitude of the reference-grid centroid
                    "latitude":  lat[valid],    # latitude of the reference-grid centroid
                    "age_group": np.full(valid.sum(), grp, dtype=object),
                    "impf_T2M":  np.full(valid.sum(), impf_id_T2M[grp], dtype=int),
                },
                geometry=gpd.points_from_xy(lon[valid], lat[valid]),
                crs="EPSG:4326",
            )
        )

gdf = pd.concat(frames, ignore_index=True) if frames else gpd.GeoDataFrame(
    columns=["value","longitude","latitude","age_group","impf_T2M","geometry"], crs="EPSG:4326"
)

# Instantiate the CLIMADA Exposures object.
from climada.entity.exposures import Exposures
exp = Exposures()
exp.set_gdf(gdf)
exp.value_unit = "people"
exp.ref_year   = year
exp.tag        = f"{CITY} FUA — WorldPop age groups ({year}) on T2M (UrbClim) grid"
exp.check()

print(exp)
if not exp.gdf.empty:
    print("Total people in exposure (T2M grid):", float(exp.gdf["value"].sum()))
    print("Breakdown by age group (T2M grid):")
    print(exp.gdf.groupby("age_group")["value"].sum().astype(float))

# Quick map of total population on the T2M centroids.
from matplotlib.colors import LogNorm
vals_all  = np.nan_to_num(pop_on_ref, nan=0.0)
valid_all = CITY_MASK
vmax = np.nanpercentile(vals_all[valid_all], 99) if valid_all.any() else 1.0
norm = LogNorm(vmin=1, vmax=max(1.0, vmax))

plt.figure(figsize=(7.8, 7.2))
sc = plt.scatter(lon[valid_all.ravel()], lat[valid_all.ravel()],
                 c=vals_all.ravel()[valid_all.ravel()], s=6, linewidths=0, alpha=0.9, norm=norm)
cb = plt.colorbar(sc); cb.set_label("People per T2M cell")
plt.title(f"{CITY} — Exposure (total people) on T2M centroids (EPSG:4326)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.gca().set_aspect('equal', 'box')
plt.tight_layout(); plt.show()

print(f"Exposure total (check): {vals_all[valid_all].sum():,.0f} people")


In [ ]:
# Sanity checks on grid shapes and centroid counts.
assert CITY_MASK.shape == (HGT, WDT)
assert lon.shape[0] == HGT * WDT and lat.shape[0] == HGT * WDT
for grp, raster in age_on_ref.items():
    assert raster.shape == (HGT, WDT)


In [ ]:
# Attach the continuous SVI value to each exposure point when the vulnerability layer is available.
if HAS_VULN:
    add_vulnerability_to_exposure_gdf(exp.gdf, svi_grid, ref_meta)

    # Fill remaining invalid values conservatively so downstream CLIMADA code does not fail.
    exp.gdf["vulnerability"] = exp.gdf["vulnerability"].replace([np.inf, -np.inf], np.nan)
    exp.gdf["vulnerability"] = exp.gdf["vulnerability"].fillna(exp.gdf["vulnerability"].median())

    print(exp.gdf["vulnerability"].describe())


## Saving Hazard and Baseline Exposure

The next block writes the baseline CLIMADA objects and a few supporting summaries. In addition to the HDF5 hazard and exposure files, the notebook exports date/event lookup tables and simple daily or annual summaries that are useful for checking what was actually encoded in the CLIMADA hazard.


In [ ]:
# Close any previously open HDF5 handles before writing CLIMADA objects.
import tables
tables.file._open_files.close_all()

# Save the baseline daily hazard and exposure objects.
haz_path = INT / f"hazard_T2M_daily_{SLUG}.h5"
exp_path = INT / f"exposure_people_{SLUG}.h5"

H.write_hdf5(haz_path)
exp.write_hdf5(exp_path)
print("Saved daily hazard →", haz_path, "| days:", H.intensity.shape[0])

# Save a row-to-date lookup because HDF5 does not preserve all event labels directly.
dates_df = pd.DataFrame({
    "row": np.arange(H.intensity.shape[0], dtype=int),
    "event_id": np.asarray(H.event_id, dtype=int),
    "date": pd.to_datetime(H.date).astype(str),
})
dates_csv = INT / f"hazard_T2M_daily_events_{SLUG}.csv"
dates_df.to_csv(dates_csv, index=False)
print("Saved events map →", dates_csv)

# Save a simple per-day summary derived from the hazard object.
# --- Two diagnostic city-mean conventions (neither feeds any downstream result;
#     deaths use per-cell H.intensity and NB05 waste-heat recomputes its own mask mean) ---
#  * mean_intensity_degC          = plain mean over ALL grid cells incl. zero off-mask cells
#                                   (diluted by tile size); the long-standing convention.
#  * mean_intensity_citymask_degC = fraction-weighted sum(I*frac)/sum(frac) = true mean over
#                                   city-mask cells only. Equal to the plain mean iff fraction==1.
num_by_event = np.asarray(H.intensity.multiply(H.fraction).sum(axis=1)).ravel().astype(float)
den_by_event = np.asarray(H.fraction.sum(axis=1)).ravel().astype(float)
mean_by_event_citymask = np.divide(
    num_by_event,
    den_by_event,
    out=np.zeros_like(num_by_event, dtype=float),
    where=den_by_event > 0,
)
mean_by_event = np.asarray(H.intensity.mean(axis=1)).ravel()  # plain mean (primary, long-standing convention)
daily_summary = pd.DataFrame({
    "row": np.arange(H.intensity.shape[0], dtype=int),
    "event_id": np.asarray(H.event_id, dtype=int),
    "date": pd.to_datetime(H.date),
    "frequency": np.asarray(H.frequency, dtype=float),
    "mean_intensity_degC": mean_by_event,
    "mean_intensity_citymask_degC": mean_by_event_citymask,
})
daily_csv = INT / f"hazard_events_T2M_daily_{SLUG}.csv"
daily_summary.to_csv(daily_csv, index=False)
print("Saved daily summary →", daily_csv)

# Also save a yearly summary aggregated from the daily hazard.
yearly_summary = (
    daily_summary
    .assign(year=lambda d: d["date"].dt.year.astype(int))
    .groupby("year", as_index=False)
    .agg(
        n_days=("row", "count"),
        citymean_degC=("mean_intensity_degC", "mean"),
        citymean_citymask_degC=("mean_intensity_citymask_degC", "mean")
    )
)
yearly_csv = INT / f"hazard_events_T2M_yearly_{SLUG}.csv"
yearly_summary.to_csv(yearly_csv, index=False)
print("Saved yearly summary →", yearly_csv)

# Optional (actually used) Track-B extreme hazard outputs 
# (heatwave/event day pathway for colder cities)
ext_cfg = cfg.get("extreme_hazard", {}) or {}
evt_cfg = ext_cfg.get("event_track", {}) or {}
if bool(ext_cfg.get("enabled", False)) and bool(ext_cfg.get("run_extreme_track", False)):
    from copy import deepcopy
    import json

    def _season_mask_by_md(dt_index, start_md: str, end_md: str):
        dt = pd.DatetimeIndex(pd.to_datetime(dt_index))
        md = dt.month.to_numpy(int) * 100 + dt.day.to_numpy(int)
        s = int(start_md.split("-")[0]) * 100 + int(start_md.split("-")[1])
        e = int(end_md.split("-")[0]) * 100 + int(end_md.split("-")[1])
        if s <= e:
            return (md >= s) & (md <= e)
        return (md >= s) | (md <= e)

    dates_ext = pd.to_datetime(daily_summary["date"])
    years_ext = dates_ext.dt.year.to_numpy(int)
    season_cfg = evt_cfg.get("season", {}) or {}
    start_md = str(season_cfg.get("start_md", "05-15"))
    end_md = str(season_cfg.get("end_md", "09-30"))
    season_mask = _season_mask_by_md(dates_ext, start_md, end_md)

    base_years = sorted({int(y) for y in cfg.get("climate", {}).get("urbclim_api", {}).get("years", list(range(2008, 2018)))})

    thr_pct = float(evt_cfg.get("threshold_percentile_default", 95))
    min_dur = int(evt_cfg.get("min_duration_default", 3))

    # Track-B threshold must come from historical warm-season climate (2008-2017 by config),
    # not from synthetic policy years.
    nc_cfg = (cfg.get("files", {}) or {}).get("t2m_mean_daily_nc", {}) or {}
    hist_pattern = str(nc_cfg.get("pattern", "T2M_year_daily_mean_*.nc"))
    dir_candidates = []
    local_hist_dir = str((cfg.get("climate", {}).get("urbclim_api", {}) or {}).get("local_dir", "")).strip()
    cfg_hist_dir = str(nc_cfg.get("dir", "")).strip()
    if local_hist_dir:
        dir_candidates.append(BASE / local_hist_dir)
    if cfg_hist_dir:
        dir_candidates.append(BASE / cfg_hist_dir)

    hist_dirs = []
    for d in dir_candidates:
        if d.exists() and d not in hist_dirs:
            hist_dirs.append(d)

    hist_files = []
    for d in hist_dirs:
        for f in sorted(d.glob(hist_pattern)):
            stem_last = f.stem.split("_")[-1]
            if stem_last.isdigit() and int(stem_last) in base_years:
                hist_files.append(f)
    hist_files = sorted({str(p.resolve()) for p in hist_files})

    if not hist_files:
        raise FileNotFoundError(
            "Track-B needs historical UrbClim files for baseline years "
            f"{base_years}, but none were found in directories: {[str(d) for d in hist_dirs]}"
        )

    with xr.open_mfdataset(hist_files, combine="by_coords", decode_times=True) as ds_hist:
        vname_hist = "T2M" if "T2M" in ds_hist.data_vars else list(ds_hist.data_vars)[0]
        da_hist = ds_hist[vname_hist]
        da_hist = da_hist.sel(time=da_hist["time.year"].isin(base_years))
        if int(da_hist.sizes.get("time", 0)) == 0:
            raise RuntimeError(
                f"Historical threshold selection is empty for years {base_years}."
            )
        space_dims = [d for d in da_hist.dims if d != "time"]
        if set(space_dims) != {"y", "x"}:
            raise ValueError(
                f"Unexpected historical T2M dims {da_hist.dims}; expected time/y/x for Track-B threshold."
            )
        mask_da = xr.DataArray(CITY_MASK.astype(bool), dims=("y", "x"))
        hist_citymean = da_hist.where(mask_da).mean(dim=("y", "x"), skipna=True).to_numpy().astype(float)
        hist_dates = pd.to_datetime(da_hist["time"].values)

    # Historical files are in Kelvin; convert to degC when needed.
    if np.nanmedian(hist_citymean) > 150:
        hist_citymean = hist_citymean - 273.15

    threshold_mode = str(evt_cfg.get("threshold_baseline_mode_default", "historical_dailymean")).strip().lower()
    if threshold_mode == "climatology_mean":
        hist_df = pd.DataFrame({
            "date": pd.to_datetime(hist_dates),
            "temp_degC": hist_citymean,
        })
        hist_df["md"] = hist_df["date"].dt.month * 100 + hist_df["date"].dt.day
        clim_by_md = hist_df.groupby("md", as_index=True)["temp_degC"].mean()
        md_vals = clim_by_md.index.to_numpy(int)
        s_md = int(start_md.split("-")[0]) * 100 + int(start_md.split("-")[1])
        e_md = int(end_md.split("-")[0]) * 100 + int(end_md.split("-")[1])
        if s_md <= e_md:
            md_mask = (md_vals >= s_md) & (md_vals <= e_md)
        else:
            md_mask = (md_vals >= s_md) | (md_vals <= e_md)
        baseline_vals = clim_by_md.to_numpy(float)[md_mask]
        baseline_years_used = sorted(pd.DatetimeIndex(hist_dates).year.unique().tolist())
        baseline_source_label = "historical climatology_mean (month/day)"
    elif threshold_mode == "historical_dailymean":
        hist_season_mask = _season_mask_by_md(pd.DatetimeIndex(hist_dates), start_md, end_md)
        baseline_vals = hist_citymean[hist_season_mask]
        baseline_years_used = sorted(pd.DatetimeIndex(hist_dates[hist_season_mask]).year.unique().tolist())
        baseline_source_label = "historical UrbClim daily means"
    else:
        raise ValueError(
            f"Unknown Track-B threshold_baseline_mode_default='{threshold_mode}'. "
            "Use one of: climatology_mean, historical_dailymean"
        )

    if baseline_vals.size == 0:
        raise RuntimeError(
            "Track-B baseline set is empty after applying warm-season window "
            f"{start_md}-{end_md} on historical years {base_years} with mode '{threshold_mode}'."
        )

    t_star = float(np.nanpercentile(baseline_vals, thr_pct))
    print(
        f"Track-B threshold from {baseline_source_label} ({min(baseline_years_used)}-{max(baseline_years_used)}): "
        f"T*=p{thr_pct:.0f} = {t_star:.2f}°C"
    )

    def _event_mask_from_threshold(city_series: np.ndarray, threshold_c: float, min_duration_days: int) -> np.ndarray:
        is_hot_local = season_mask & (city_series > float(threshold_c))
        grp_local = np.cumsum(np.r_[True, is_hot_local[1:] != is_hot_local[:-1]])
        run_len_local = pd.Series(is_hot_local).groupby(grp_local).transform("size").to_numpy(int)
        return is_hot_local & (run_len_local >= int(min_duration_days))

    city_series = daily_summary["mean_intensity_degC"].to_numpy(float)
    is_event = _event_mask_from_threshold(city_series, t_star, min_dur)

    # Sparsity safeguard for colder cities:
    # keep heatwave duration central first; only relax duration after trying threshold percentiles.
    min_event_days_required = int(evt_cfg.get("min_event_days_required", 3))
    if int(np.sum(is_event)) < min_event_days_required:
        thr_cfg = [float(x) for x in evt_cfg.get("threshold_percentile_options", [thr_pct])]
        dur_cfg = [int(x) for x in evt_cfg.get("min_duration_options", [min_dur])]
        thr_options = [float(thr_pct)] + [float(x) for x in thr_cfg if float(x) != float(thr_pct)]
        dur_options = [int(min_dur)] + [int(x) for x in dur_cfg if int(x) != int(min_dur)]

        raw_pairs = []
        # First: vary percentile at central duration (e.g., p95->p93 with dur=3)
        for cand_thr in thr_options:
            raw_pairs.append((float(cand_thr), int(min_dur)))
        # Then: relax duration (e.g., dur=2) for each percentile option
        for cand_dur in dur_options:
            if int(cand_dur) == int(min_dur):
                continue
            for cand_thr in thr_options:
                raw_pairs.append((float(cand_thr), int(cand_dur)))

        candidate_pairs = []
        seen_pairs = set()
        for pair in raw_pairs:
            if pair in seen_pairs:
                continue
            seen_pairs.add(pair)
            candidate_pairs.append(pair)

        tried = []
        selected = False
        for cand_thr, cand_dur in candidate_pairs:
            cand_t_star = float(np.nanpercentile(baseline_vals, cand_thr))
            cand_event = _event_mask_from_threshold(city_series, cand_t_star, cand_dur)
            cand_count = int(np.sum(cand_event))
            tried.append((cand_thr, cand_dur, cand_t_star, cand_count))
            if cand_count >= min_event_days_required:
                thr_pct = float(cand_thr)
                min_dur = int(cand_dur)
                t_star = float(cand_t_star)
                is_event = cand_event
                selected = True
                break
        if tried:
            print("Track-B sparsity check candidates (pct, min_dur, T*, event_days):")
            for _pct, _dur, _t, _n in tried:
                print(f"  p{_pct:.0f}, dur={_dur}: T*={_t:.2f}°C, event_days={_n}")
            if selected:
                print(
                    f"→ Adopted fallback Track-B setting: p{thr_pct:.0f}, min_duration={min_dur}, "
                    f"T*={t_star:.2f}°C"
                )
            else:
                print("→ No sparsity fallback reached the required event-day count; keeping central settings.")

    exceed_city = np.clip(city_series - t_star, 0.0, None)
    event_city = np.where(is_event, exceed_city, 0.0)

    daily_ext = daily_summary.copy()
    daily_ext["is_event_day"] = is_event.astype(int)
    daily_ext["event_intensity_citymean_degC"] = event_city
    daily_ext_csv = INT / f"hazard_events_T2M_daily_{SLUG}_extreme.csv"
    daily_ext.to_csv(daily_ext_csv, index=False)

    yearly_ext = (
        daily_ext.assign(year=pd.to_datetime(daily_ext["date"]).dt.year.astype(int))
        .groupby("year", as_index=False)
        .agg(
            n_days=("row", "count"),
            n_event_days=("is_event_day", "sum"),
            mean_event_intensity_citymean_degC=("event_intensity_citymean_degC", "mean"),
            max_event_intensity_citymean_degC=("event_intensity_citymean_degC", "max"),
        )
    )
    yearly_ext_csv = INT / f"hazard_events_T2M_yearly_{SLUG}_extreme.csv"
    yearly_ext.to_csv(yearly_ext_csv, index=False)

    H_ext = deepcopy(H)
    I_ext = H.intensity.tocsr().astype("float32").copy()
    I_ext.data = np.clip(I_ext.data - t_star, 0.0, None)
    row_ids = np.repeat(np.arange(I_ext.shape[0]), np.diff(I_ext.indptr))
    I_ext.data *= is_event[row_ids].astype("float32")
    I_ext.eliminate_zeros()
    H_ext.intensity = I_ext
    H_ext.units = "degC exceedance above heatwave threshold"

    haz_ext_path = INT / f"hazard_T2M_daily_{SLUG}_extreme.h5"
    H_ext.write_hdf5(haz_ext_path)
    dates_ext_csv = INT / f"hazard_T2M_daily_events_{SLUG}_extreme.csv"
    dates_df.to_csv(dates_ext_csv, index=False)

    meta = {
        "track": "extreme",
        "threshold_percentile": thr_pct,
        "threshold_degC": t_star,
        "min_duration_days": min_dur,
        "season_start_md": start_md,
        "season_end_md": end_md,
        "threshold_baseline_source": threshold_mode,
        "threshold_baseline_source_label": baseline_source_label,
        "baseline_years_configured": sorted(base_years),
        "baseline_years_used": baseline_years_used,
        "event_days_total": int(np.sum(is_event)),
    }
    with open(INT / f"hazard_extreme_meta_{SLUG}.json", "w") as f:
        json.dump(meta, f, indent=2)

    import shutil
    haz_ext_out = OUT / haz_ext_path.name
    haz_ext_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(haz_ext_path, haz_ext_out)

    print("Saved extreme daily summary →", daily_ext_csv)
    print("Saved extreme yearly summary →", yearly_ext_csv)
    print("Saved extreme hazard →", haz_ext_path)
    print("Copied extreme hazard →", haz_ext_out)


In [ ]:
# Save the same baseline CLIMADA objects again and copy them to OUT for easier downstream access.
haz_path = INT / f"hazard_T2M_daily_{SLUG}.h5"
exp_path = INT / f"exposure_people_{SLUG}.h5"
H.write_hdf5(haz_path)
exp.write_hdf5(exp_path)
print("Saved daily hazard →", haz_path, "| days:", H.intensity.shape[0])

# Also copy the files to OUT so later notebooks can find them easily.
import shutil
haz_path_out = OUT / f"hazard_T2M_daily_{SLUG}.h5"
exp_path_out = OUT / f"exposure_people_{SLUG}.h5"
haz_path_out.parent.mkdir(parents=True, exist_ok=True)
exp_path_out.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(haz_path, haz_path_out)
shutil.copy2(exp_path, exp_path_out)
print("Copied to OUT →", haz_path_out)
print("Copied to OUT →", exp_path_out)


In [ ]:
# Build a compact yearly T2M grid summary from the daily hazard.
import numpy as np
import pandas as pd
import rasterio as rio

# Load the grid shape from the reference template.
tmpl_path = INT / "template_ref.tif"
with rio.open(tmpl_path) as src:
    HGT, WDT = src.height, src.width

n_days, n_cells = H.intensity.shape
assert n_cells == HGT * WDT, f"Cells {n_cells} != HGT*WDT = {HGT*WDT}"

# Year per event
dates = pd.to_datetime(pd.Series(H.date))
years = dates.dt.year.to_numpy().astype(int)
uniq_years = np.unique(years)

# For each year, compute mean daily T2M per cell and reshape back to 2D.
DATA = np.empty((len(uniq_years), HGT, WDT), dtype="float32")
for k, yr in enumerate(uniq_years):
    mask = (years == yr)
    # Mean over all days in that year => one value per cell
    arr_1d = np.asarray(H.intensity[mask].mean(axis=0), dtype="float32")
    DATA[k, :, :] = arr_1d.reshape(HGT, WDT)

# Save the result as a compact NumPy archive.
t2m_npz_path = INT / f"t2m_hazard_2020_2050_{SLUG}.npz"
np.savez_compressed(t2m_npz_path, years=uniq_years, data=DATA)
print("Saved yearly T2M hazard grid →", t2m_npz_path)


In [ ]:
# Save the baseline exposure carrying the attached vulnerability values for later scaling.

from climada.entity.exposures import Exposures
import shutil

base_exp_path = INT / f"exposure_with_vulnerability_{SLUG}.h5"
exp.write_hdf5(base_exp_path)
print("Saved baseline exposure-with-vulnerability →", base_exp_path)

# Also copy the file to OUT for easier downstream reuse.
base_exp_out = OUT / base_exp_path.name
shutil.copy2(base_exp_path, base_exp_out)
print("Copied to OUT →", base_exp_out)


## Future Exposure Objects

Future CLIMADA exposures are generated from the scenario/year grids saved in NB02. The scaling is done age group by age group and cell by cell, so the baseline row structure is preserved while the `value` field is updated to match the future population totals. The vulnerability column, when present, is kept static because NB03 does not model future changes in social vulnerability.


In [ ]:
from cityheat.dynamic_vulnerability import build_dynamic_vulnerability_and_exposures

dyn_enabled = bool(cfg.get("vulnerability", {}).get("dynamic", {}).get("enabled", False))
if dyn_enabled:
    print("Refreshing projected vulnerability baselines and exposure-with-vulnerability files...")
    dyn_vuln = build_dynamic_vulnerability_and_exposures(
        cfg=cfg,
        int_dir=INT,
        out_dir=OUT,
        base_path=BASE,
        ref_meta=ref_meta,
        city_mask=CITY_MASK,
        verbose=True,
    )
    print(f"Dynamic vulnerability keys built: {sorted(dyn_vuln.keys())}")

    # Keep the projected 2020 baseline SVI in memory for diagnostics below.
    vuln = load_vulnerability(INT, SLUG, year=2020)
    svi_grid = vuln["svi"]
    print(f"Updated projected 2020 SVI mean={np.nanmean(svi_grid):.3f}, std={np.nanstd(svi_grid):.3f}")
else:
    dyn_vuln = {}
    print("Dynamic vulnerability disabled in config; skipping projected vulnerability build.")
    vuln = load_vulnerability(INT, SLUG)
    svi_grid = vuln["svi"]
    print(f"Loaded legacy baseline SVI mean={np.nanmean(svi_grid):.3f}, std={np.nanstd(svi_grid):.3f}")


## Projected Vulnerability Layer

This notebook no longer treats social vulnerability as static after the baseline year. Instead, it builds a **time-varying 100 m vulnerability layer** on the same reference grid used for hazard and exposure, and then writes year- and scenario-specific `exposure_with_vulnerability` files that downstream notebooks read directly.

### How the projected layer is built

The projected SVI keeps the original high-resolution spatial pattern from the city baseline and updates it over time with two external trend anchors:

- **Baseline high-resolution structure (100 m):** the original city layer is built from the same three dimensions used in the static workflow: thermal vulnerability from `GHS_AGE`, foreign-born share from Census 2021 (`OTH / T`), and non-employment from Census 2021 (`1 - EMP / Y_1564`).
- **Short-run regional change (to 2030):** DRMKC Risk Data Hub NUTS3 indicators provide the near-term trend signal used to shift the city-level means of the social components.
- **Long-run national scenario envelope (2030 to 2050):** the GDL / GVI SSP projections provide the country-level long-term trajectory used to continue the projection under each SSP.

The social components are projected on an **absolute-comparable scale**, not by re-ranking each year. This matters because it lets a cell become more or less vulnerable over time in a way that is comparable across years, instead of only preserving within-year ranking.

At the grid level, the projection preserves the baseline intra-city anomaly pattern and applies year-specific mean shifts with a persistence parameter (`k`) and a spatial-retention parameter (`phi`). Thermal vulnerability evolves more slowly through retrofit and new-build assumptions, while the social components follow the DRMKC + GVI bridge.

### What gets written to disk

Running the dynamic vulnerability block creates:

- `vulnerability_<slug>_2020.npz`
- `vulnerability_<slug>_2030.npz`
- `vulnerability_<slug>_<scenario>_2040.npz`
- `vulnerability_<slug>_<scenario>_2050.npz`
- matching `exposure_with_vulnerability` HDF5 files for each direct year and scenario year

These exposure files are the key interface with the rest of the framework: later notebooks load them instead of reusing one static vulnerability column.

### How later notebooks use projected vulnerability

Within the main March 2026 workflow:

- **NB03** builds the projected vulnerability rasters and refreshes the exposure files.
- **NB04** reads the year/scenario-specific exposure-with-vulnerability files together with the matching hazard years.
- **NB05** now uses the correct year-specific SVI layer for vulnerability stratification and equity diagnostics.
- **NB07** now uses the correct focus-year / scenario SVI layer for vegetation equity targeting and disparity analysis.

So in the main impact workflow, **hazard, exposure, and vulnerability now evolve consistently over time**.

### Important exception

One block in `NB05` intentionally keeps vulnerability fixed: the `climate_only_frozen_vulnerability_2020` diagnostic. That comparison is designed to isolate the effect of changing climate while holding social vulnerability at the baseline year, so it remains frozen by construction.


In [ ]:
# Diagnostic check: summarize and visualize projected vulnerability across the main anchor years.
from cityheat.vulnerability_diagnostics import build_projected_vulnerability_diagnostics

if dyn_enabled:
    vuln_diag_df, vuln_diag_fig = build_projected_vulnerability_diagnostics(
        cfg=cfg,
        int_dir=INT,
        out_dir=OUT,
        scenario=cfg.get("exp_scenario", "SSP2"),
        ref_meta=ref_meta,
        city_mask=CITY_MASK,
        verbose=False,
    )
    print("Projected vulnerability diagnostics summary:")
    display(vuln_diag_df.round(3))
    display(vuln_diag_fig)
else:
    print("Dynamic vulnerability disabled in config; skipping projected vulnerability diagnostics.")
